# FedProx with PyTorch ResNet50 on PlantVillage

This Kaggle notebook trains a 5-client FedProx simulation using an ImageFolder dataset rooted at `/kaggle/input/plantvillage`. It runs 50 communication rounds, 10 local epochs per selected client, mixed precision training, and writes round-level Accuracy, Precision, Recall, and F1 metrics to CSV.

In [ ]:
import copy
import csv
import gc
import math
import os
import random
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
class CFG:
    data_root = Path('/kaggle/input')
    preferred_subset = 'grayscale'  # This Kaggle dataset also includes color and segmented folders.
    output_dir = Path('/kaggle/working')
    seed = 42

    num_clients = 5
    communication_rounds = 50
    local_epochs = 10
    client_fraction = 1.0

    image_size = 224
    batch_size = 32
    num_workers = 2
    lr = 1e-4
    weight_decay = 1e-4
    fedprox_mu = 0.01

    train_ratio = 0.80
    val_ratio = 0.10
    test_ratio = 0.10

    use_imagenet_weights = False  # Set True only if Kaggle internet or cached torchvision weights are available.
    freeze_backbone = False
    amp = True

    metrics_csv = output_dir / 'fedprox_resnet50_metrics.csv'
    final_model_path = output_dir / 'fedprox_resnet50_final.pth'


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(CFG.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CFG.output_dir.mkdir(parents=True, exist_ok=True)
print('Device:', device)

In [ ]:
IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def immediate_image_count(path: Path) -> int:
    return sum(1 for f in path.iterdir() if f.is_file() and f.suffix.lower() in IMG_EXTENSIONS)


def looks_like_imagefolder(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    class_dirs = [p for p in path.iterdir() if p.is_dir()]
    if len(class_dirs) < 2:
        return False
    # Require images directly inside class folders. This avoids treating folders like
    # color/grayscale/segmented as labels when the real classes are one level deeper.
    valid_class_dirs = sum(1 for class_dir in class_dirs if immediate_image_count(class_dir) > 0)
    return valid_class_dirs >= 2


def find_imagefolder_root(root: Path, preferred_subset: str = 'grayscale') -> Path:
    input_root = Path('/kaggle/input')
    search_roots = []
    for candidate_root in [root, input_root / 'plantvillage', input_root / 'plantvillage-dataset']:
        if candidate_root.exists() and candidate_root not in search_roots:
            search_roots.append(candidate_root)
    if input_root.exists():
        for mounted_dataset in input_root.iterdir():
            if mounted_dataset.is_dir() and mounted_dataset not in search_roots:
                search_roots.append(mounted_dataset)
    if not search_roots:
        raise FileNotFoundError('No Kaggle input folders found. Add the PlantVillage dataset from the right-side Add data panel first.')

    candidates = []
    for search_root in search_roots:
        candidates.extend([search_root] + [p for p in search_root.rglob('*') if p.is_dir()])
    preferred = [p for p in candidates if p.name.lower() == preferred_subset.lower() and looks_like_imagefolder(p)]
    if preferred:
        preferred.sort(key=lambda p: (len(p.parts), str(p)))
        return preferred[0]

    valid = [p for p in candidates if looks_like_imagefolder(p)]
    if not valid:
        available = []
        if input_root.exists():
            available = [str(p) for p in input_root.iterdir() if p.is_dir()]
        raise FileNotFoundError(
            'No ImageFolder-style class directory found. Expected something like '
            '/kaggle/input/<dataset>/grayscale/Apple___Apple_scab/*.JPG. '
            f'Available /kaggle/input folders: {available}'
        )
    valid.sort(key=lambda p: (len(p.parts), str(p)))
    return valid[0]


imagefolder_root = find_imagefolder_root(CFG.data_root, CFG.preferred_subset)
print('Using ImageFolder root:', imagefolder_root)

train_tfms = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((CFG.image_size, CFG.image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

base_dataset = datasets.ImageFolder(imagefolder_root)
class_names = base_dataset.classes
num_classes = len(class_names)
targets = np.array(base_dataset.targets)

print('Classes:', num_classes)
print('Images:', len(base_dataset))
print(pd.DataFrame(Counter(targets).items(), columns=['class_idx', 'count']).sort_values('class_idx').head())

In [ ]:
class TransformSubset(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = list(indices)
        self.transform = transform
        self.targets = [dataset.targets[i] for i in self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        path, target = self.dataset.samples[self.indices[idx]]
        image = Image.open(path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, target


def stratified_split_indices(targets, train_ratio=0.8, val_ratio=0.1, seed=42):
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for cls in np.unique(targets):
        cls_idx = np.where(targets == cls)[0]
        rng.shuffle(cls_idx)
        n = len(cls_idx)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        train_idx.extend(cls_idx[:n_train])
        val_idx.extend(cls_idx[n_train:n_train + n_val])
        test_idx.extend(cls_idx[n_train + n_val:])
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)
    return train_idx, val_idx, test_idx


def make_stratified_client_indices(train_indices, targets, num_clients, seed=42):
    rng = np.random.default_rng(seed)
    client_indices = [[] for _ in range(num_clients)]
    train_indices = np.array(train_indices)
    train_targets = targets[train_indices]
    for cls in np.unique(train_targets):
        cls_train_indices = train_indices[train_targets == cls].copy()
        rng.shuffle(cls_train_indices)
        shards = np.array_split(cls_train_indices, num_clients)
        for client_id, shard in enumerate(shards):
            client_indices[client_id].extend(shard.tolist())
    for indices in client_indices:
        rng.shuffle(indices)
    return client_indices


train_indices, val_indices, test_indices = stratified_split_indices(
    targets, CFG.train_ratio, CFG.val_ratio, CFG.seed
)
client_indices = make_stratified_client_indices(train_indices, targets, CFG.num_clients, CFG.seed)

val_dataset = TransformSubset(base_dataset, val_indices, eval_tfms)
test_dataset = TransformSubset(base_dataset, test_indices, eval_tfms)
client_datasets = [TransformSubset(base_dataset, idxs, train_tfms) for idxs in client_indices]

val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)
client_loaders = [
    DataLoader(ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, pin_memory=True, drop_last=False)
    for ds in client_datasets
]

print('Train / Val / Test:', len(train_indices), len(val_indices), len(test_indices))
for i, idxs in enumerate(client_indices):
    print(f'Client {i}: {len(idxs)} samples')

In [ ]:
def build_resnet50(num_classes: int) -> nn.Module:
    if CFG.use_imagenet_weights:
        weights = models.ResNet50_Weights.DEFAULT
    else:
        weights = None
    model = models.resnet50(weights=weights)
    if CFG.freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


def get_trainable_named_params(model: nn.Module):
    return [(name, param) for name, param in model.named_parameters() if param.requires_grad]


def fedprox_loss(model: nn.Module, global_params, mu: float, base_loss):
    if mu <= 0:
        return base_loss
    prox = torch.zeros((), device=next(model.parameters()).device)
    for name, param in get_trainable_named_params(model):
        prox = prox + torch.sum((param - global_params[name]) ** 2)
    return base_loss + (mu / 2.0) * prox


def local_train(global_model, loader, client_id: int):
    local_model = copy.deepcopy(global_model).to(device)
    local_model.train()

    global_params = {
        name: param.detach().clone().to(device)
        for name, param in get_trainable_named_params(global_model.to(device))
    }

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, local_model.parameters()), lr=CFG.lr, weight_decay=CFG.weight_decay)
    scaler = GradScaler(enabled=(CFG.amp and device.type == 'cuda'))
    running_loss = 0.0
    steps = 0

    progress = tqdm(range(CFG.local_epochs), desc=f'Client {client_id} local epochs', leave=False)
    for _ in progress:
        epoch_loss = 0.0
        epoch_steps = 0
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=(CFG.amp and device.type == 'cuda')):
                logits = local_model(images)
                ce_loss = criterion(logits, labels)
                loss = fedprox_loss(local_model, global_params, CFG.fedprox_mu, ce_loss)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += loss.item()
            running_loss += loss.item()
            epoch_steps += 1
            steps += 1
        progress.set_postfix(loss=epoch_loss / max(1, epoch_steps))

    local_state = {k: v.detach().cpu() for k, v in local_model.state_dict().items()}
    avg_loss = running_loss / max(1, steps)

    del local_model, optimizer, scaler, global_params
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return local_state, len(loader.dataset), avg_loss


def aggregate_fedavg(local_states, sample_counts):
    total_samples = float(sum(sample_counts))
    aggregated = copy.deepcopy(local_states[0])
    for key in aggregated.keys():
        if torch.is_floating_point(aggregated[key]):
            aggregated[key] = sum(state[key] * (count / total_samples) for state, count in zip(local_states, sample_counts))
        else:
            aggregated[key] = local_states[0][key]
    return aggregated

In [ ]:
@torch.no_grad()
def evaluate(model, loader, split_name='val'):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    for images, labels in tqdm(loader, desc=f'Evaluating {split_name}', leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast(enabled=(CFG.amp and device.type == 'cuda')):
            logits = model(images)
            loss = criterion(logits, labels)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        total_loss += loss.item() * labels.size(0)

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0
    )
    return {
        'loss': total_loss / max(1, len(loader.dataset)),
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }


def append_metrics_csv(path: Path, row: dict):
    exists = path.exists()
    with path.open('a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)

In [ ]:
global_model = build_resnet50(num_classes).to(device)

if CFG.metrics_csv.exists():
    CFG.metrics_csv.unlink()

num_selected = max(1, int(CFG.num_clients * CFG.client_fraction))
round_history = []
best_f1 = -1.0
best_state = None

for round_idx in range(1, CFG.communication_rounds + 1):
    round_start = time.time()
    selected_clients = sorted(random.sample(range(CFG.num_clients), num_selected))
    print(f'\nRound {round_idx}/{CFG.communication_rounds} | clients: {selected_clients}')

    local_states = []
    sample_counts = []
    local_losses = []

    for client_id in selected_clients:
        state, n_samples, local_loss = local_train(global_model, client_loaders[client_id], client_id)
        local_states.append(state)
        sample_counts.append(n_samples)
        local_losses.append(local_loss)

    aggregated_state = aggregate_fedavg(local_states, sample_counts)
    global_model.load_state_dict(aggregated_state)
    global_model.to(device)

    val_metrics = evaluate(global_model, val_loader, 'val')
    elapsed = time.time() - round_start
    row = {
        'round': round_idx,
        'split': 'val',
        'selected_clients': '|'.join(map(str, selected_clients)),
        'train_samples': int(sum(sample_counts)),
        'mean_local_loss': float(np.mean(local_losses)),
        'loss': val_metrics['loss'],
        'accuracy': val_metrics['accuracy'],
        'precision_macro': val_metrics['precision'],
        'recall_macro': val_metrics['recall'],
        'f1_macro': val_metrics['f1'],
        'round_seconds': elapsed,
    }
    append_metrics_csv(CFG.metrics_csv, row)
    round_history.append(row)

    if val_metrics['f1'] > best_f1:
        best_f1 = val_metrics['f1']
        best_state = copy.deepcopy(global_model.state_dict())

    print(
        f"val_acc={val_metrics['accuracy']:.4f} "
        f"val_precision={val_metrics['precision']:.4f} "
        f"val_recall={val_metrics['recall']:.4f} "
        f"val_f1={val_metrics['f1']:.4f} "
        f"time={elapsed/60:.1f}m"
    )

    del local_states, aggregated_state
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

if best_state is not None:
    global_model.load_state_dict(best_state)

torch.save({
    'model_state_dict': global_model.state_dict(),
    'classes': class_names,
    'config': {k: str(v) if isinstance(v, Path) else v for k, v in CFG.__dict__.items() if not k.startswith('__') and not callable(v)},
}, CFG.final_model_path)

print('Saved metrics to:', CFG.metrics_csv)
print('Saved model to:', CFG.final_model_path)

In [ ]:
test_metrics = evaluate(global_model, test_loader, 'test')
test_row = {
    'round': 'final_test',
    'split': 'test',
    'selected_clients': 'all',
    'train_samples': len(train_indices),
    'mean_local_loss': '',
    'loss': test_metrics['loss'],
    'accuracy': test_metrics['accuracy'],
    'precision_macro': test_metrics['precision'],
    'recall_macro': test_metrics['recall'],
    'f1_macro': test_metrics['f1'],
    'round_seconds': '',
}
append_metrics_csv(CFG.metrics_csv, test_row)

print('Final test metrics')
print(pd.DataFrame([test_metrics]))
pd.read_csv(CFG.metrics_csv).tail()

## Outputs

- Metrics CSV: `/kaggle/working/fedprox_resnet50_metrics.csv`
- Final model checkpoint: `/kaggle/working/fedprox_resnet50_final.pth`

The CSV contains validation metrics for every communication round plus a final row with held-out test metrics.